# 03 · Data Acquisition — FieldPlant

Downloads FieldPlant (Moupojou et al., 2023) from the authors' official Roboflow
project, **version 11** (5,156 field photos, no augmentation, CC BY 4.0), and crops
each annotated leaf into a single-leaf image in the PlantVillage format.

- Only labels that match a project class unambiguously are kept
  (`config.FIELDPLANT_CLASS_MAP`): maize gray leaf spot, healthy maize, tomato late
  blight (*mildiou*), healthy tomato. Every other label is dropped and recorded.
- Each crop records its source photo, so crops from one photo can be kept in the
  same partition when the data is split.
- The download goes to Colab's local disk; only the crops are saved to Drive.

Needs the `ROBOFLOW_API_KEY` Colab secret.

In [4]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

Already up to date.
## main...origin/main
 M ml/notebooks/03_fieldplant_acquisition.ipynb


In [6]:
%pip install -q -r {ML_DIR}/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 143.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.2 MB/s eta 0:00:00


## Download version 11 to local disk

In [7]:
from google.colab import userdata

from src.data import fieldplant

coco_root = fieldplant.download(userdata.get("ROBOFLOW_API_KEY"), Path("/content/fieldplant_raw"))
print("Downloaded FieldPlant version", fieldplant.VERSION, "to", coco_root)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/fieldplant_raw in coco:: 100%|██████████| 5160/5160 [00:34<00:00, 148.98it/s]


Downloaded FieldPlant version 11 to /content/fieldplant_raw


## Crop the mapped leaves to Drive

Starts from an empty FieldPlant folder so a re-run never leaves stale crops behind.
Boxes smaller than `config.MIN_IMAGE_SIZE` are skipped and counted as dropped.

In [ ]:
import shutil

import pandas as pd

if config.FIELDPLANT_DIR.exists():
    shutil.rmtree(config.FIELDPLANT_DIR)

rows, dropped = fieldplant.crop_leaves(
    coco_root, config.FIELDPLANT_CLASS_MAP, config.FIELDPLANT_DIR, config.MIN_IMAGE_SIZE
)
crops = pd.DataFrame(rows)

print("Kept crops per class:")
print(crops["class"].value_counts().to_string())
print("\nTotal kept:", len(crops), "| from", crops["source_image"].nunique(), "source photos")

missing = set(config.FIELDPLANT_CLASS_MAP) - set(crops["fieldplant_label"])
assert not missing, f"Mapped labels produced no crops - check spelling in config: {missing}"

## What was dropped

In [ ]:
dropped_df = (
    pd.DataFrame(dropped.items(), columns=["fieldplant_label", "boxes"])
    .sort_values("boxes", ascending=False)
    .reset_index(drop=True)
)
print("Dropped boxes:", dropped_df["boxes"].sum())
display(dropped_df)

## Save the inventory

`fieldplant_crops.csv` lists every crop with its class, original label and source
photo; `fieldplant_dropped.csv` records every dropped label for the data report.

In [ ]:
crops.to_csv(config.METADATA_DIR / "fieldplant_crops.csv", index=False)
dropped_df.to_csv(config.METADATA_DIR / "fieldplant_dropped.csv", index=False)
print("Saved fieldplant_crops.csv and fieldplant_dropped.csv to", config.METADATA_DIR)

## Save to GitHub

Commits the changed notebook to a new branch and prints a link to open the pull
request (`main` is protected). Change the branch name if this cell has already been
run once. Save the notebook (Ctrl+S) before running this cell.

In [ ]:
from src import github_sync

print(github_sync.status(REPO_DIR))
print()
print(github_sync.push(
    REPO_DIR,
    branch="colab/fieldplant-acquisition",
    message="Acquire FieldPlant leaves for the mapped classes",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
))